In [1]:
import cv2
import mediapipe as mp
import pickle
import numpy as np
import time
import sys
from pathlib import Path

In [2]:
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import utils as utils_module


# left_model_filename = r'../Models/left_model.p'
# right_model_filename = r'../Models/right_model.p'
# pose_model_filename = r'../Models/pose_model.p'
left_model_filename = r'./Models/left_model.p'
right_model_filename = r'./Models/right_model.p'
pose_model_filename = r'./Models/pose_model.p'


# Function to load a model from a pickle file
def load_model(filename):
    with open(filename, 'rb') as f:
        model_data = pickle.load(f)
        return model_data['model']

# Loading the pre-trained models
left_model = load_model(left_model_filename)
right_model = load_model(right_model_filename)
pose_model = load_model(pose_model_filename)

utils = utils_module.Utils()

print("Models loaded successfully.")

Models loaded successfully.


In [3]:
# Initialize Mediapipe hands and pose solutions
mp_hands = mp.solutions.hands
mp_pose = mp.solutions.pose

hands = mp_hands.Hands(
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)
pose = mp_pose.Pose(
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

# Define coordinate axes for angle calculations
axes = {
    "x": np.array([1, 0, 0]),
    "-x": np.array([-1, 0, 0]),
    "y": np.array([0, 1, 0]),
    "-y": np.array([0, -1, 0]),
    "z": np.array([0, 0, 1]),
    "-z": np.array([0, 0, -1]),
}


# Global previous positions and timing for rest detection
prev_positions = {
    "left_wrist": None,
    "right_wrist": None,
    "left_shoulder": None,
    "right_shoulder": None,
    "timestamp": None,
    # per-landmark counters for consecutive low-speed frames
    "counters": {
        "left_wrist": 0,
        "right_wrist": 0,
        "left_shoulder": 0,
        "right_shoulder": 0
    }
}


In [7]:
def calulating_percentage(avg, all_classes):
    """
    Given a list of average probabilities (avg) and corresponding class labels (all_classes),
    returns a list of percentage values adjusted by class-specific thresholds.
    """
    
    individual_threshold = {
        "sun": 0.9, "help": 0.9, "teacher": 0.9, "support": 0.9,
        "paper": 0.9, "love": 0.9, "dance": 0.9, "water": 0.9,
        "accident": 0.9, "yes": 0.9, "thick": 0.9, "high": 0.9,
        "poor": 0.9, "i": 0.9, "my": 0.9, "important_1": 0.9,
        "important_2": 0.9, "deaf": 0.9, "winner": 0.9, "eat": 0.9,
        "pizza": 0.9, "go": 0.9, "isl": 0.9, "friend": 0.9,
        "school": 0.9, "deep": 0.9, "loud": 0.9, "flat": 0.9,
        "slow": 0.9, "sad": 0.9, "soft": 0.9, "happy": 0.9,
        "poot": 0.9, "quiet": 0.9, "book": 0.9, "woman": 0.9
    }

    threshold_percentage = []
    for score, cls in zip(avg, all_classes):
        # Use class-specific threshold to adjust the final percentage
        threshold_val = individual_threshold.get(cls.lower(), 1.0)
        threshold_percentage.append(score * 100 / threshold_val)

    return threshold_percentage

In [ ]:
frame_count: int = 0
accumulated_prob: float = None
final_pred_text: str = ""
final_prob_text: str = ""

rest_buffer_frame: int = 0

REST_SPEED_THRESHOLD :float = 30.0  
MIN_POINTS_FOR_REST :int   = 2       
REQUIRED_CONSECUTIVE_FRAMES :int   = 3  
REST_DELAY:int = 3

USE_REST: bool = False

words_hist:list = []

cap = cv2.VideoCapture(1)

fps = cap.get(cv2.CAP_PROP_FPS)      
slow_factor:int = 6                        


while True:
    ret, frame = cap.read()
    
    frame = cv2.flip(frame, 1)
    
    if not ret:
        break

    # Preparing the frame for Mediapipe
    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    res_hands = hands.process(image_rgb)
    res_pose = pose.process(image_rgb)
    annotated = frame.copy()

    # left_prediction, right_prediction, pose_prediction = None, None, None
    left_probs, right_probs, pose_probs = None, None, None

    try:
        if USE_REST:
            resting = utils.is_resting(res_hands, annotated.shape, REST_DELAY)
        else:
            resting = False
        if not resting:
            # Hand landmarks
            if res_hands.multi_hand_landmarks:
                for hand_landmarks, handedness in zip(res_hands.multi_hand_landmarks, res_hands.multi_handedness):
                    label = handedness.classification[0].label
                    features = utils.extract_hand_features(
                        hand_landmarks.landmark,
                        res_pose.pose_landmarks.landmark if res_pose.pose_landmarks else []
                    )

                    if label == 'Left':
                        left_probs = left_model.predict_proba([features])[0]
                    elif label == 'Right':
                        right_probs = right_model.predict_proba([features])[0]

                    # Draw hand landmarks on the frame
                    mp.solutions.drawing_utils.draw_landmarks(
                        frame, hand_landmarks, mp.solutions.hands.HAND_CONNECTIONS
                    )

            # Pose landmarks
            if res_pose.pose_landmarks:
                mp.solutions.drawing_utils.draw_landmarks(
                    frame, res_pose.pose_landmarks, mp.solutions.pose.POSE_CONNECTIONS
                )
                pose_landmarks = res_pose.pose_landmarks.landmark
                pose_features = utils.extract_pose_features(pose_landmarks)
                pose_prediction = pose_model.predict([pose_features])[0]
                pose_probs = pose_model.predict_proba([pose_features])[0]

            # Gathering all class labels used by the three models
            all_classes = sorted(
                set(left_model.classes_).union(
                set(right_model.classes_)).union(
                set(pose_model.classes_))
            )

            # Aligning probabilities with the master list of classes
            left_probs_aligned = np.zeros(len(all_classes))
            right_probs_aligned = np.zeros(len(all_classes))
            pose_probs_aligned = np.zeros(len(all_classes))

            if left_probs is not None:
                left_dict = dict(zip(left_model.classes_, left_probs))
                left_probs_aligned = np.array([left_dict.get(cls, 0) for cls in all_classes]) * 100

            if right_probs is not None:
                right_dict = dict(zip(right_model.classes_, right_probs))
                right_probs_aligned = np.array([right_dict.get(cls, 0) for cls in all_classes]) * 100

            if pose_probs is not None:
                pose_dict = dict(zip(pose_model.classes_, pose_probs))
                pose_probs_aligned = np.array([pose_dict.get(cls, 0) for cls in all_classes]) * 100

            # Determine the number of available sources (hand(s)/pose)
            num_sources = sum(prob is not None for prob in [left_probs, right_probs, pose_probs])
            avg = (left_probs_aligned + right_probs_aligned + pose_probs_aligned) / (100 * num_sources if num_sources else 1)

            # Convert these averages into final percentages based on custom thresholds
            avg_probs = calulating_percentage(avg, all_classes)

            if accumulated_prob is None:
                accumulated_prob = np.zeros_like(avg_probs)
            accumulated_prob += avg_probs
            frame_count += 1
        
        # Updating pred to rest and reset frame accumulation
        else:
            final_pred_text = "Final Prediction: Resting" 
            accumulated_prob = None
            frame_count = 0
    
    except Exception as e:
        print(f"Error processing frame: {e}")
        status_text = "Error"
        color = (0, 0, 255)
        cv2.putText(annotated, status_text, (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2, cv2.LINE_AA)

    # Updating the final prediction text after a fixed no. of frames
    if frame_count == 5:
        max_idx = np.argmax(accumulated_prob)
        max_class = all_classes[max_idx]
        if max_class != "Resting":
            words_hist.append(max_class)
        final_pred_text = f"Final Prediction: {max_class}" 
        final_prob_text = f"Prob: {accumulated_prob[max_idx]:.2f}"
        accumulated_prob = None
        frame_count = 0

    
    if final_pred_text:
        cv2.putText(
            frame,
            final_pred_text,
            (10, 30),
            cv2.FONT_HERSHEY_DUPLEX,
            0.8,
            (255, 0, 0),
            2
        )
        cv2.putText(
            frame,
            final_prob_text,
            (10, 60),
            cv2.FONT_HERSHEY_DUPLEX,
            0.6,
            (55, 214, 106),
            2
        )
    
    
    cv2.putText(
        frame,
        f"{words_hist[-5:]}",
        (10, 90),
        cv2.FONT_HERSHEY_DUPLEX,
        0.6,
        (55, 214, 106),
        2
    )
        
    cv2.imshow("Hand and Pose Tracking", frame)
    
    # time.sleep(1 / (fps / slow_factor))
    
    # Press 'q' to quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break


cap.release()
cv2.destroyAllWindows()